In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [30]:
def load_dataset(name):
    if name == "Iris":
        data = load_iris(as_frame=True)
        return data.frame
    elif name == "Wine":
        data = load_wine(as_frame=True)
        return data.frame
    elif name == "Titanic":
        return sns.load_dataset("titanic")

In [31]:
dataset_selector = widgets.Dropdown(
    options=["Iris", "Wine", "Titanic"],
    description="Zbiór danych:"
)

In [32]:
def run_app(dataset_name):
    df = load_dataset(dataset_name)
    display(Markdown(f"# Auto-EDA – {dataset_name}"))
    display(Markdown("## Podgląd danych"))
    display(df.head())
    display(Markdown("## Typy danych"))
    dtypes_df = pd.DataFrame({
        "Kolumna": df.columns,
        "Typ danych": df.dtypes.astype(str)
    })
    display(dtypes_df.reset_index(drop=True))

    display(Markdown("## Statystyki opisowe"))
    numeric_df = df.select_dtypes(include=np.number)
    display(numeric_df.describe())
    display(Markdown("## Braki danych"))

    missing = df.isnull().sum()
    percent = (missing / len(df)) * 100

    missing_df = pd.DataFrame({
        "Braki": missing,
        "Procent [%]": percent.round(2)
    })

    missing_df = missing_df[missing_df["Braki"] > 0]

    if missing_df.empty:
        display(Markdown("**Brak brakujących wartości w zbiorze danych.**"))
    else:
        display(missing_df)

    plt.figure(figsize=(8, 4))
    sns.heatmap(df.isnull(), cbar=False)
    plt.title("Heatmapa braków danych")
    plt.show()

    display(Markdown("## Outliery (IQR)"))
    numeric_cols = df.select_dtypes(include=np.number).drop(columns=["target"], errors="ignore")
    Q1 = numeric_cols.quantile(0.25)
    Q3 = numeric_cols.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((numeric_cols < (Q1 - 1.5 * IQR)) |
                (numeric_cols > (Q3 + 1.5 * IQR)))
    outlier_count = outliers.sum()
    display(outlier_count)

    display(Markdown("## Wizualizacja wybranych zmiennych"))
    display(Markdown("**Uwaga:** Aby wyświetlić macierz korelacji należy wybrać więcej niż jedną zmienną korzystając z przycisku control"))

    col_selector = widgets.SelectMultiple(
        options=numeric_cols.columns,
        description="Kolumny:",
        layout=widgets.Layout(width="50%")
    )

    def plot_selected(cols):
        if not cols:
            return

        for col in cols:
            plt.figure(figsize=(6, 4))
            sns.histplot(df[col], kde=True)
            plt.title(f"Rozkład zmiennej: {col}")
            plt.show()

            plt.figure(figsize=(6, 4))
            sns.boxplot(x=df[col])
            plt.title(f"Boxplot: {col}")
            plt.show()

        if len(cols) > 1:
            plt.figure(figsize=(6, 5))
            sns.heatmap(df[list(cols)].corr(), annot=True, cmap="coolwarm")
            plt.title("Macierz korelacji")
            plt.show()

    widgets.interact(plot_selected, cols=col_selector)

In [33]:
widgets.interact(run_app, dataset_name=dataset_selector)

interactive(children=(Dropdown(description='Zbiór danych:', options=('Iris', 'Wine', 'Titanic'), value='Iris')…

<function __main__.run_app(dataset_name)>